In [3]:
"""
Benchmark 2: Image-Only Model
Tests whether MRI scans alone can predict AD conversion

FIXES vs previous version:
- [CRITICAL] Survival time now anchored to MCI diagnosis, not study
  baseline. Previously used df["Years_bl"].iloc[ad_idx] (absolute),
  which made image-only survival times systematically longer than all
  other methods and corrupted every RMST/KM/C-index comparison.
  Fix: subtract df["Years_bl"].iloc[mci_idx] from both event and
  censoring times, matching Benchmarks 1, 3, and the thesis.
- [CRITICAL] StandardScaler now fitted on training patient indices only
  and applied to all sequences before export. The thesis applies a
  scaler to tabular features; for fairness the image encoder output
  (which mixes learned weights and raw pixel statistics) is likewise
  standardised using train statistics only.
- [MINOR]   manifest["PTID"] cast to str to match VALID_PATIENTS types.

Architecture:
- ResNet18 image encoder
- LSTM for temporal modelling
- NO tabular features
"""

import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import numpy as np
import os
import pickle
from sklearn.preprocessing import StandardScaler
from lifelines.utils import concordance_index
import warnings
warnings.filterwarnings('ignore')

CONFIG = {
    'img_out_dim':  256,
    'lstm_hidden':  128,
    'lstm_layers':  2,
    'dropout':      0.3,
    'lr':           5e-4,
    'weight_decay': 1e-4,
    'epochs':       100,
    'batch_size':   16,
    'max_seq_len':  10,
    'device': 'cuda' if torch.cuda.is_available() else 'cpu'
}

# ============================================================================
# DATASET
# Raw image features stored without scaling; scaler applied externally
# after the train/val split so val statistics never touch the scaler fit.
# ============================================================================

class ImageOnlyDataset(Dataset):
    def __init__(self, manifest, valid_patients, transform=None, max_seq_len=10):
        self.sequences   = []
        self.transform   = transform
        self.max_seq_len = max_seq_len
        self.scaler      = None   # set via apply_scaler() after split

        manifest = manifest.copy()
        manifest["path"] = manifest["path"].str.replace("\\", "/", regex=False)
        manifest["path"] = "./AD_Multimodal/TFN_AD/" + manifest["path"]

        processed          = 0
        skipped_not_valid  = 0

        for ptid in manifest["PTID"].unique():
            if ptid not in valid_patients:
                skipped_not_valid += 1
                continue

            try:
                patient_rows = manifest[manifest["PTID"] == ptid]
                if len(patient_rows) == 0:
                    continue

                df = pd.read_pickle(patient_rows.iloc[0]["path"])

                dx_seq = df["DX"].tolist()
                if "MCI" not in dx_seq:
                    continue

                mci_idx = dx_seq.index("MCI")
                ad_idx  = next(
                    (i for i, x in enumerate(dx_seq[mci_idx + 1:],
                                             start=mci_idx + 1)
                     if x in ["AD", "Dementia"]),
                    -1
                )

                # FIX: anchor to MCI diagnosis (was absolute Years_bl)
                if ad_idx != -1:
                    time_to_event = (df["Years_bl"].iloc[ad_idx]
                                     - df["Years_bl"].iloc[mci_idx])
                    event = 1
                else:
                    time_to_event = (df["Years_bl"].iloc[-1]
                                     - df["Years_bl"].iloc[mci_idx])
                    event = 0

                imgs, times = [], []
                valid_visits = 0

                for _, visit in df.iterrows():
                    image_path = visit["image_path"].replace(
                        "/home/mason/ADNI_Dataset/",
                        "./AD_Multimodal/ADNI_Dataset/"
                    )
                    if not os.path.exists(image_path):
                        continue

                    img = Image.open(image_path).convert("RGB")
                    if self.transform:
                        img = self.transform(img)

                    imgs.append(img)
                    times.append(float(visit["Years_bl"]))
                    valid_visits += 1

                    if valid_visits >= max_seq_len:
                        break

                if valid_visits < 2:
                    continue

                pad_len = max_seq_len - valid_visits
                for _ in range(pad_len):
                    imgs.append(torch.zeros_like(imgs[-1]))
                    times.append(times[-1])

                self.sequences.append({
                    'ptid':          ptid,
                    'imgs':          torch.stack(imgs),
                    'times':         np.array(times, dtype=np.float32),
                    'seq_len':       valid_visits,
                    'time_to_event': float(time_to_event),
                    'event':         int(event),
                    # Placeholder for image encoder outputs; filled by
                    # apply_scaler() after the model's first-pass encoding.
                    # For the LSTM input we use the raw tensor — scaling
                    # of the exported CSV features is handled at export time.
                })
                processed += 1

            except Exception as e:
                print(f"  Warning: skipped patient {ptid}: {e}")
                continue

        print(f"  Processed             : {processed} valid patients")
        print(f"  Skipped (not in set)  : {skipped_not_valid}")

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        return (
            seq['imgs'],
            seq['times'],
            seq['seq_len'],
            seq['time_to_event'],
            seq['event'],
            seq['ptid'],
        )


# ============================================================================
# MODEL
# ============================================================================

class ImageOnlyModel(nn.Module):
    def __init__(self, config):
        super().__init__()

        base = models.resnet18(pretrained=True)
        for param in list(base.parameters())[:-20]:
            param.requires_grad = False
        self.features = nn.Sequential(*list(base.children())[:-1])
        self.img_proj = nn.Linear(512, config['img_out_dim'])

        self.lstm = nn.LSTM(
            input_size  = config['img_out_dim'],
            hidden_size = config['lstm_hidden'],
            num_layers  = config['lstm_layers'],
            batch_first = True,
            dropout     = config['dropout'] if config['lstm_layers'] > 1 else 0,
            bidirectional = True
        )

        self.risk_head = nn.Sequential(
            nn.Linear(config['lstm_hidden'] * 2, 64),
            nn.ReLU(),
            nn.Dropout(config['dropout']),
            nn.Linear(64, 1)
        )

    def encode_image(self, img):
        feats = self.features(img)
        return self.img_proj(feats.view(feats.size(0), -1))

    def forward(self, img_seq, seq_lengths):
        bs, seq_len = img_seq.shape[:2]
        img_feats = [self.encode_image(img_seq[:, t]) for t in range(seq_len)]
        img_seq_feat = torch.stack(img_feats, dim=1)

        packed = nn.utils.rnn.pack_padded_sequence(
            img_seq_feat, seq_lengths.cpu(),
            batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)
        h_final = torch.cat([h_n[-2], h_n[-1]], dim=1)
        return self.risk_head(h_final)


# ============================================================================
# LOSS & TRAINING
# ============================================================================

def cox_loss(risk_scores, times, events):
    order       = torch.argsort(times, descending=True)
    risk_scores = risk_scores[order]
    events      = events[order]
    log_risk    = risk_scores.view(-1)
    log_cumsum  = torch.logcumsumexp(log_risk, dim=0)
    loss        = -(log_risk - log_cumsum) * events
    return loss.sum() / (events.sum() + 1e-7)


def train_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    for batch in loader:
        imgs, times, seq_lens, t_event, event, _ = batch
        imgs    = imgs.to(device)
        seq_lens = torch.LongTensor(seq_lens)
        t_event = t_event.float().to(device)
        event   = event.float().to(device)

        risk_scores = model(imgs, seq_lens)
        loss = cox_loss(risk_scores.squeeze(), t_event, event)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)


def validate(model, loader, device):
    model.eval()
    all_risks, all_times, all_events = [], [], []
    with torch.no_grad():
        for batch in loader:
            imgs, times, seq_lens, t_event, event, _ = batch
            imgs     = imgs.to(device)
            seq_lens = torch.LongTensor(seq_lens)
            risks    = model(imgs, seq_lens)
            all_risks.extend(risks.cpu().numpy().flatten())
            all_times.extend(t_event.numpy())
            all_events.extend(event.numpy())

    all_events = np.array(all_events).astype(bool)
    return concordance_index(
        np.array(all_times), -np.array(all_risks), all_events
    )


# ============================================================================
# EXPORT
# FIX: scaler fitted on train encoder outputs, applied before CSV write.
# ============================================================================

def export_features(model, full_loader, train_indices, device, output_path):
    """
    Export first-visit image features for each patient.

    A StandardScaler is fitted on the encoder outputs of training patients
    only, then applied to all patients before writing to CSV. This mirrors
    the scaler treatment in the thesis pipeline and ensures that test-patient
    statistics do not influence the scaling.
    """
    model.eval()

    # --- Pass 1: collect raw encoder outputs for ALL patients ---
    raw_feats = {}   # ptid -> np.array of shape (img_out_dim,)
    meta      = {}   # ptid -> (Years_bl, time_to_event, event)

    with torch.no_grad():
        for batch in full_loader:
            imgs, times, seq_lens, t_event, event, ptids = batch
            imgs = imgs.to(device)

            for i in range(len(ptids)):
                feat = model.encode_image(imgs[i:i + 1, 0])
                raw_feats[ptids[i]] = feat[0].cpu().numpy()
                meta[ptids[i]] = (
                    float(times[i][0]),
                    float(t_event[i]),
                    int(event[i])
                )

    all_ptids = list(raw_feats.keys())

    # Map dataset indices back to PTIDs for scaler fitting
    # train_indices are indices into full_loader.dataset.sequences
    train_ptids = set(
        full_loader.dataset.sequences[i]['ptid']
        for i in train_indices
        if i < len(full_loader.dataset.sequences)
    )

    # --- Fit scaler on TRAIN patients only ---
    train_feats = np.vstack([
        raw_feats[p] for p in all_ptids if p in train_ptids
    ])
    scaler = StandardScaler()
    scaler.fit(train_feats)
    print(f"  Scaler fitted on {len(train_feats)} training patients")

    # --- Scale ALL patients and write CSV ---
    rows = []
    for ptid in all_ptids:
        scaled = scaler.transform(raw_feats[ptid].reshape(1, -1))[0]
        years_bl, tte, ev = meta[ptid]
        row = {
            "PTID":          ptid,
            "Years_bl":      years_bl,
            "time_to_event": tte,
            "event":         ev,
        }
        for k, val in enumerate(scaled):
            row[f"img_feat_{k}"] = float(val)
        rows.append(row)

    df = pd.DataFrame(rows).sort_values(["PTID", "Years_bl"])
    df.to_csv(output_path, index=False)

    print(f"\n  Exported to {output_path}")
    print(f"  Patients        : {df['PTID'].nunique()}")
    print(f"  Image features  : {len([c for c in df.columns if c.startswith('img_feat_')])}")
    return df


# ============================================================================
# MAIN
# ============================================================================

def main():
    print("=" * 80)
    print("BENCHMARK 2: IMAGE-ONLY MODEL")
    print("=" * 80)

    device = CONFIG['device']
    print(f"\nDevice: {device}")

    print("\nLoading valid patient list...")
    with open('VALID_PATIENTS.pkl', 'rb') as f:
        VALID_PATIENTS = pickle.load(f)
    VALID_PATIENTS = set(str(p) for p in VALID_PATIENTS)
    print(f"Valid patients: {len(VALID_PATIENTS)}")

    manifest = pd.read_csv("./AD_Multimodal/TFN_AD/AD_Patient_Manifest.csv")
    manifest["PTID"] = manifest["PTID"].astype(str)

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])

    dataset = ImageOnlyDataset(
        manifest, VALID_PATIENTS, transform,
        max_seq_len=CONFIG['max_seq_len']
    )
    print(f"Total sequences: {len(dataset)}")

    n_train = int(0.8 * len(dataset))
    n_val   = len(dataset) - n_train
    train_dataset, val_dataset = torch.utils.data.random_split(
        dataset, [n_train, n_val],
        generator=torch.Generator().manual_seed(42)
    )
    train_indices = list(train_dataset.indices)

    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                              shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_dataset,   batch_size=CONFIG['batch_size'],
                              shuffle=False, num_workers=0)

    model     = ImageOnlyModel(CONFIG).to(device)
    optimizer = torch.optim.AdamW(model.parameters(),
                                  lr=CONFIG['lr'],
                                  weight_decay=CONFIG['weight_decay'])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )

    print("\nTraining...")
    best_c_index    = 0.0
    patience_counter = 0

    for epoch in range(CONFIG['epochs']):
        train_loss  = train_epoch(model, train_loader, optimizer, device)
        val_c_index = validate(model, val_loader, device)
        scheduler.step(val_c_index)

        print(f"Epoch {epoch + 1}/{CONFIG['epochs']} "
              f"— Loss: {train_loss:.4f}, C-index: {val_c_index:.4f}")

        if val_c_index > best_c_index:
            best_c_index = val_c_index
            torch.save(model.state_dict(), 'image_only_model.pth')
            patience_counter = 0
            print(f"  New best: {best_c_index:.4f}")
        else:
            patience_counter += 1

        if patience_counter >= 15:
            print("Early stopping")
            break

    # Export with scaler fitted on train only
    model.load_state_dict(
        torch.load('image_only_model.pth', weights_only=False)
    )
    full_loader = DataLoader(dataset, batch_size=CONFIG['batch_size'],
                             shuffle=False, num_workers=0)

    export_features(
        model, full_loader, train_indices, device,
        "image_only_features.csv"
    )

    print("\n" + "=" * 80)
    print(f"Best C-index: {best_c_index:.4f}")
    print("Survival time anchored to MCI diagnosis")
    print("Scaler fitted on train patients only")
    print("=" * 80)


if __name__ == "__main__":
    main()

BENCHMARK 2: IMAGE-ONLY MODEL

Device: cuda

Loading valid patient list...
Valid patients: 161
  Processed             : 161 valid patients
  Skipped (not in set)  : 221
Total sequences: 161

Training...
Epoch 1/100 — Loss: 2.2963, C-index: 0.5728
  New best: 0.5728
Epoch 2/100 — Loss: 2.2354, C-index: 0.5243
Epoch 3/100 — Loss: 1.8318, C-index: 0.5696
Epoch 4/100 — Loss: 1.5313, C-index: 0.4434
Epoch 5/100 — Loss: 1.8852, C-index: 0.5761
  New best: 0.5761
Epoch 6/100 — Loss: 1.5332, C-index: 0.5566
Epoch 7/100 — Loss: 1.5535, C-index: 0.5016
Epoch 8/100 — Loss: 1.3288, C-index: 0.6440
  New best: 0.6440
Epoch 9/100 — Loss: 1.3159, C-index: 0.5599
Epoch 10/100 — Loss: 1.3512, C-index: 0.6311
Epoch 11/100 — Loss: 1.1023, C-index: 0.6408
Epoch 12/100 — Loss: 1.2084, C-index: 0.5987
Epoch 13/100 — Loss: 0.9258, C-index: 0.5502
Epoch 14/100 — Loss: 1.0846, C-index: 0.5372
Epoch 15/100 — Loss: 1.0924, C-index: 0.5016
Epoch 16/100 — Loss: 0.8009, C-index: 0.4854
Epoch 17/100 — Loss: 0.6346,